In [ ]:
!pip install -U cir-model -qq

In [ ]:
sub_name='ensemble_ridge'

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import copy
import os
import gc
from cir_model import CenteredIsotonicRegression 
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from scipy.special import expit
from tqdm import tqdm
pd.set_option('display.max_columns', 500)

train = pd.read_csv("/kaggle/input/playground-series-s6e1/train.csv")
true = train['exam_score'].values
train.head()

In [ ]:
files = []
x_train = []
x_test = []
PATH = "/kaggle/input/pg-s6e1-oofs/"

print("Loading files...")
for root, dirs, files_ in os.walk(PATH):
    for file in files_:
        filepath = os.path.join(root, file)
        if 'oof' in filepath[27:-4] or 'preds' in filepath[27:-4] or 'ensemble' in filepath[27:-4]: continue
        
        print(f"=> {filepath[27:-4]} ",end="")
        print()
        oof = pd.read_csv(filepath[:-4] + '_oof.csv')
        if 'id' in oof.columns.tolist():
            oof = oof.drop('id', axis=1)
        x_train.append(oof.values.flatten())
        files.append(filepath[27:-4])
        df = pd.read_csv(filepath)
        if 'id' in df.columns.tolist():
            df = df.drop('id', axis=1)
        pred = df.values.flatten()
        x_test.append(pred)
        

In [ ]:
x_train = np.stack(x_train).T
print("Our combined OOF have shape:",x_train.shape)

x_test = np.stack(x_test).T
print("Our combined PRED have shape:",x_test.shape)

In [ ]:
kf = KFold(n_splits=5, random_state=42, shuffle=True)
for i in tqdm(range(x_train.shape[1])):
    cir_model = CenteredIsotonicRegression()
    cir_model.fit(x_train[:, i], true)
    pred = cir_model.transform(x_train[:, i])
    nan_mask = np.isnan(pred)
    if np.any(nan_mask):
        pred[nan_mask] = x_train[:, i][nan_mask]
    x_train[:, i] = pred
    pred = cir_model.transform(x_test[:, i])
    nan_mask = np.isnan(pred)
    if np.any(nan_mask):
        pred[nan_mask] = x_test[:, i][nan_mask]
    x_test[:, i] = pred

In [ ]:
x_train_df = pd.DataFrame(x_train, columns=files).drop([], axis=1)
corr_matrix = np.corrcoef(x_train_df.values, rowvar=False)

plt.figure(figsize=(28, 25))
sns.heatmap(corr_matrix,
            cmap='coolwarm',
            xticklabels=x_train_df.columns,
            yticklabels=x_train_df.columns,
            annot=True,
            square=True)

plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.title("Correlation Matrix (All Feature Labels)")
plt.tight_layout()
plt.show()

In [ ]:
x_train_df = pd.DataFrame(x_test, columns=files).drop([], axis=1)
corr_matrix = np.corrcoef(x_train_df.values, rowvar=False)

plt.figure(figsize=(28, 25))
sns.heatmap(corr_matrix,
            cmap='coolwarm',
            xticklabels=x_train_df.columns,
            yticklabels=x_train_df.columns,
            annot=True,
            square=True)

plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.title("Correlation Matrix (All Feature Labels)")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import RidgeCV

ridge = RidgeCV(alphas=[1])
ridge.fit(x_train, true)
y_pred = ridge.predict(x_train)
print(np.sqrt(mean_squared_error(true, y_pred)))

In [ ]:
model_names = copy.deepcopy(files)
model_scores = []

for file in files:
    df = pd.read_csv(f'/kaggle/input/pg-s6e1-oofs/{file}_oof.csv')
    if 'id' in df.columns.tolist():
        df = df.drop('id', axis=1)
    pred = df.values.flatten()
    score = np.sqrt(mean_squared_error(true, pred))
    model_scores.append(score)
model_names.append('ensemble')
model_scores.append(np.sqrt(mean_squared_error(true, y_pred)))

idx = np.argsort(model_scores)
model_names = [model_names[i] for i in idx]
model_scores = [model_scores[i] for i in idx]

plt.figure(figsize=(20, 25))
bars = plt.barh(model_names, model_scores)
for bar, score in zip(bars, model_scores):
    plt.text(min(score + 0.00005, 8.71 + 0.00005), bar.get_y() + bar.get_height() / 2,
             f'{score:.6f}', va='center')
plt.xlim(8.565, 8.71)
plt.xlabel('Score')
plt.title('OOF Scores')
plt.grid(True)
plt.show()

In [ ]:
model_names = files
weights = ridge.coef_
idx = np.argsort(weights)
model_names = [model_names[i] for i in idx]
weights = weights[idx]

plt.figure(figsize=(20, 20))
plt.barh(model_names, weights)
plt.xlabel('Weight')
plt.title('Ridge Ensemble Weights')
plt.grid(True)
plt.show()

In [ ]:
kf = KFold(n_splits=5, random_state=42, shuffle=True)

cir_model = CenteredIsotonicRegression()
cir_model.fit(y_pred, true)
pred = cir_model.transform(y_pred)
nan_mask = np.isnan(pred)
if np.any(nan_mask):
    pred[nan_mask] = y_pred[nan_mask]
y_pred = pred

In [ ]:
pred_ = ridge.predict(x_test)

In [ ]:
pred = cir_model.transform(pred_)
nan_mask = np.isnan(pred)
if np.any(nan_mask):
    pred[nan_mask] = pred_[nan_mask]
pred_ = pred

In [ ]:
print(np.sqrt(mean_squared_error(true, y_pred)))

In [ ]:
submission = pd.read_csv('/kaggle/input/playground-series-s6e1/sample_submission.csv')
submission['exam_score'] = pred_
submission.to_csv(f'{sub_name}.csv', index=False)